# Employee Attrition Machine Learning

This notebook supports two business goals: identify factors associated with employees leaving, and improve retention in high-risk departments and job roles. The model estimates an employee's probability of attrition so HR can prioritize supportive interventions.

Hypotheses explored include job role, promotion history, monthly income, overtime, satisfaction, work-life balance, travel, distance from home, tenure, stock options, and department.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / 'ml_model.py').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, roc_auc_score
from ml_model import load_data, feature_columns, train_model, save_model

data = load_data()
data.head()

In [ ]:
print(f"Rows: {len(data):,}; columns: {data.shape[1]}")
print(data['Attrition'].value_counts(normalize=True).mul(100).round(2))
display(data.groupby('JobRole')['AttritionBinary'].agg(['count', 'mean']).sort_values('mean', ascending=False).style.format({'mean': '{:.1%}'}))
display(data.groupby('Department')['AttritionBinary'].agg(['count', 'mean']).style.format({'mean': '{:.1%}'}))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=data, x='AttritionBinary', y='MonthlyIncome', ax=axes[0], errorbar=None)
sns.barplot(data=data, x='AttritionBinary', y='YearsSinceLastPromotion', ax=axes[1], errorbar=None)
sns.barplot(data=data, x='AttritionBinary', y='JobSatisfaction', ax=axes[2], errorbar=None)
for axis in axes:
    axis.set_xlabel('Attrition (0 = No, 1 = Yes)')
plt.tight_layout()

## Train and evaluate a risk model

The preprocessing is fitted only on the training split. Categorical variables are one-hot encoded, numeric variables are scaled, and class balancing addresses the minority attrition class. Employee number and the target/duplicate label are excluded to prevent leakage.

In [ ]:
model, x_test, y_test, metrics = train_model(data)
print(f"Accuracy: {metrics['accuracy']:.3f}")
print(f"ROC-AUC: {metrics['roc_auc']:.3f}")
print(classification_report(metrics['y_test'], metrics['predictions'], target_names=['Stayed', 'Left']))
ConfusionMatrixDisplay.from_predictions(metrics['y_test'], metrics['predictions'], display_labels=['Stayed', 'Left'], cmap='Blues')
plt.title('Attrition classification results')
plt.show()
save_model(model)

In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = pd.DataFrame({'Feature': feature_names, 'Coefficient': model.named_steps['classifier'].coef_[0]})
display(coefficients.assign(Absolute=lambda frame: frame['Coefficient'].abs()).sort_values('Absolute', ascending=False).head(20).drop(columns='Absolute'))

In [ ]:
scored = data.copy()
scored['Attrition probability'] = model.predict_proba(scored[feature_columns(data)])[:, 1]
scored['Risk band'] = pd.cut(scored['Attrition probability'], bins=[-0.01, 0.30, 0.60, 1.0], labels=['Low', 'Medium', 'High'])
display(scored.sort_values('Attrition probability', ascending=False)[['EmployeeNumber', 'Department', 'JobRole', 'Attrition probability', 'Risk band']].head(10))